<div style="padding: 20px 24px; border-radius: 12px; background: linear-gradient(110deg, #12304a, #187b8f); color: white; margin-bottom: 16px;">
  <div style="font-size: 0.78em; letter-spacing: 0.12em; text-transform: uppercase; opacity: 0.85;">HydroMT-SFINCS | Example 05</div>
  <h1 style="color: white; margin: 8px 0 4px;">SFINCS results: determine maximum water depth</h1>
  <div style="opacity: 0.95;">Compare maximum-water-depth workflows for regular and subgrid models.</div>
</div>


This notebook contains 2 examples, one for a standard regular SFINCS model, and one for a subgrid SFINCS model.

<div style="padding: 14px 18px; border-left: 4px solid #d99000; background: #fff7e6; border-radius: 6px; margin: 12px 1em 12px 0; width: calc(100% - 1em); box-sizing: border-box; color: #111111;"><strong><span style="font-size: 1.15em; margin-right: 4px;">&#9888;&#65039;</span>IMPORTANT NOTE</strong>, the methods for producing a flood depth map for the regular vs subgrid versions of SFINCS are <strong>NOT</strong> the same, and <strong>NOT</strong> interchangeable!</div>

Contents:
1. Determine maximum water depth for a regular model
2. Determine maximum water depth for a subgrid model

Each method is organized into numbered steps, with matching headings below.


# 1. Determine maximum water depth for a regular model

### Steps

1. Read model results and inspect the model layout.
2. Optionally write maximum water depth to a GeoTIFF.
3. Mask permanent water and apply a minimum depth threshold.
4. Plot maximum water depth.


The first example used in this notebook is based on a regular SFINCS model, i.e. no subgrid tables are used. In the absence of the subgrid tables, SFINCS computes the water depth by simply substracting the bed levels from the water levels. The (maximum) water depth **h(max)** is stored in the NetCDF output (*sfincs_map.nc*). 

How to derive maximum water depths for a model including subgrid tables is explained [later](#2.-Determine-maximum-water-depth-for-a-subgrid-model) in this notebook.

In [ ]:
from os.path import join
import matplotlib.pyplot as plt

from hydromt_sfincs import SfincsModel, utils
from hydromt_sfincs.workflows import downscale_floodmap

### Step 1: Read model results

The model results in sfincs_map.nc are saved as in a staggered grid format, see [SGRID convention](http://sgrid.github.io/sgrid/). Here we show how to retrieve the face values and translate the dimensions from node indices (m, n) to (x, y) coordinates in order to plot the results on a map.

In [ ]:
sfincs_root = "sfincs_compound"  # (relative) path to sfincs root
mod = SfincsModel(sfincs_root, data_libs=["artifact_data"], mode="r")

In [ ]:
# we can simply read the model results (sfincs_map.nc and sfincs_his.nc) using the output.read method
mod.output.read()

# the following variables have been found
list(mod.output.data.keys())

Plot the model layout


In [ ]:
fig, ax = mod.plot_basemap(fn_out=None, bmap="sat", figsize=(11, 7), zoomlevel=12)

### Step 2: Write maximum waterdepth to geotiff file (optional)

<div style="padding: 14px 18px; border-left: 4px solid #187b8f; background: #eaf6f8; border-radius: 6px; margin: 12px 1em 12px 0; width: calc(100% - 1em); box-sizing: border-box; color: #111111;"><strong><span style="font-size: 1.15em; margin-right: 4px;">&#128161;</span> NOTE</strong> When writing a DataArray, the mask is optional; without one, the full array is written. Pass <code>mask=mod.grid.data["mask"]</code> to restrict the output to active model cells.</div>

In [ ]:
# from hydromt_sfincs import writers
# hmax = mod.output.data["hmax"].max(dim="timemax")
# hmax.attrs.update(long_name="maximum water depth", unit="m")
# writers.write_raster(
#     hmax,
#     join(mod.root.path, "gis"),
#     compress="LZW",
# )

### Step 3: Prepare maximum water depth with a surface water mask

First we mask the water depth based on a map of permanent water to get the flood extent. 
The mask is calculated from the Global Surface Water Occurence (GSWO) dataset. 

In [ ]:
# read global surface water occurance (GSWO) data to mask permanent water
gswo = mod.data_catalog.get_rasterdataset("gswo", geom=mod.region, buffer=10)

# permanent water where water occurence > 5%
gswo_mask = gswo.raster.reproject_like(mod.grid.data, method="max") <= 5

# hmax is computed by SFINCS and read-in from the sfincs_map.nc file
da_hmax = mod.output.data["hmax"].max(dim="timemax")

# get overland flood depth with GSWO and set minimum flood depth
da_hmax = da_hmax.where(gswo_mask)

# update attributes for colorbar label later
da_hmax.attrs.update(long_name="flood depth", unit="m")

<div style="padding: 14px 18px; border-left: 4px solid #187b8f; background: #eaf6f8; border-radius: 6px; margin: 12px 1em 12px 0; width: calc(100% - 1em); box-sizing: border-box; color: #111111;"><strong><span style="font-size: 1.15em; margin-right: 4px;">&#9888;&#65039;</span> NOTE</strong> To read data for a different region than Northen Italy add this data to the data catalog:<pre><code>mod.data_catalog.from_yml('/path/to/data_catalog.yml')</code></pre></div>

### Step 4: Plot maximum water depth

Here we plot the maximum water depth on top of the `plot_basemap` method to also include the locations of discharge source points and observation gauge locations.

In [ ]:
# minimum flood depth [m] to plot
hmin = 0.05
da_hmax = da_hmax.where(da_hmax > hmin)

In [ ]:
fig, ax = mod.plot_basemap(
    fn_out=None,
    figsize=(8, 6),
    variable=da_hmax,
    plot_bounds=False,
    plot_geoms=False,
    bmap="sat",
    zoomlevel=12,
    vmin=0,
    vmax=2.0,
    cmap=plt.cm.viridis,
    cbar_kwargs={"shrink": 0.6, "anchor": (0, 0)},
)
ax.set_title(f"SFINCS maximum water depth")
# plt.savefig(join(mod.root, 'figs', 'hmax.png'), dpi=225, bbox_inches="tight")

# 2. Determine maximum water depth for a subgrid model

When subgrid tables are used, these are based on elevation data of a resolution higher than the resolution of the computational grid. The bed levels stored in the NetCDF output (*zb* in *sfincs_map.nc*) are the minimum bed levels from the subgrid tables of each computational cell. Therefore, to properly derive water depths, the higher resolution elevation data should be used instead of simply using the bed levels from the model output (this would result in an overestimation of flood extents). The process of interpolating the water levels onto the higher-resolution elevation data is called downscaling.

The downscaling of the floodmap roughly includees the following steps:

1. Select your high-resolution elevation dataset
2. Retrieve the maximum water levels from your model
3. Determine if and how you want to mask your floodmap
4. Write a geotiff of your downscaled floodmap
5. Plot your downscaled floodmap here, or in QGIS

In [ ]:
# select the example model, `sfincs_root` is the (relative) path to sfincs root.
sfincs_root = "sfincs_compound"
mod = SfincsModel(sfincs_root, data_libs=["artifact_data"], mode="r")

### Step 1: Select the high-resolution elevation dataset

In the example, we used *GEBCO* (~450m resolution) and *MERIT Hydro* (~90m resolution) to create a model on 50 meters resolution. In this case, including subgrid tables didn't add much information to the model. To still illustrate the process of downscaling, we will use the *dep.tif* (in the gis folder) which has 50m resolution. 

When creating a subgrid with `setup_subgrid`, you can easily create a geotiff on the subgrid resolution in the `subgrid` folder with the argument `write_dep_tif=True`.

In [ ]:
# first we are going to select our highest-resolution elevation dataset
depfile = join(sfincs_root, "gis", "dep.tif")

# with the depfile on subgrid resolution this would be:
# depfile = join(sfincs_root, "subgrid", "dep_subgrid.tif")

# read the data
da_dep = mod.data_catalog.get_rasterdataset(depfile)

### Step 2: Retrieve maximum water levels from the model

In [ ]:
# now we are reading in the model results
mod.output.read()

# now assuming we have a subgrid model, we don't have hmax available, so we are using zsmax (maximum water levels)
# compute the maximum over all time steps
da_zsmax = mod.output.data["zsmax"].max(dim="timemax")

### Step 3: Determine how to mask the floodmap

Again, we have to determine how we want to mask the floodmap. We could use the GSWO dataset to mask permanent water in a similar way as above. Alternatively we could use a geodataframe (e.g. OpenStreetMap landareas) to mask water bodies and again we can (also) use a threshold to mask minimum flood depth.

In [ ]:
# Here, we load the OpenStreetMap land areas geodataframe
gdf_osm = mod.data_catalog.get_geodataframe("osm_landareas")

# And we set a minimum water depth threshold for masking
hmin = 0.05

<div style="padding: 14px 18px; border-left: 4px solid #187b8f; background: #eaf6f8; border-radius: 6px; margin: 12px 1em 12px 0; width: calc(100% - 1em); box-sizing: border-box; color: #111111;"><strong>NOTE:</strong> small rivers are not masked with this geodataframe</div>

### Step 4: Downscale the floodmap

Now we have specified the water levels, the high-resolution bed levels and how we want to mask the flood depths, we can finally create the downscaled floodmap


In [ ]:
da_hmax = downscale_floodmap(
    zsmax=da_zsmax,
    dep=da_dep,
    hmin=hmin,
    gdf_mask=gdf_osm,
    # floodmap_fn=join(sfincs_root, "floodmap.tif") # uncomment to save to <mod.root>/floodmap.tif
)

### Step 5: Plot the downscaled floodmap

Finally, we can create a basemap plot with hmax on top.


In [ ]:
fig, ax = mod.plot_basemap(
    fn_out=None,
    figsize=(8, 6),
    variable=da_hmax,
    plot_bounds=False,
    plot_geoms=False,
    bmap="sat",
    zoomlevel=11,
    vmin=0,
    vmax=3.0,
    cbar_kwargs={"shrink": 0.6, "anchor": (0, 0)},
)
ax.set_title(f"SFINCS maximum water depth")
# plt.savefig(join(mod.root, 'figs', 'hmax.png'), dpi=225, bbox_inches="tight")